In [95]:
from utils import *

n_sim = 1000
n = int(1000.5/0.7)
B_RF = int(10_000 )

seed = 42
n_covariates = 3

X_pred_point = pd.DataFrame({'X_1': [40], 'X_2': [1], 'X_3': [80]})
data_generation_parameter =    { 'shape_weibull': 1,  'p_1': -0.4, 'p_2': -0.2, 'p_3': -0.1, 'n': n,
                                    'scale_weibull_base':   5_329_174_608_059        , 
                                    'rate_censoring':       0.04477    , 
                                    'tau': 20,
                                    'X_pred_point': X_pred_point}  



params_rf =         {   'n_estimators':B_RF,                        
                        'max_depth':4,
                        'min_samples_split':5,
                        'max_features': 'log2',
                        'random_state':  seed,
                        'weighted_bootstrapping': True, }

data = create_weibull_data(params=data_generation_parameter, random_state=seed)
df_train, df_test = stratified_split(data=data, random_state=seed, test_size=0.3)
df_train = create_data_with_ipc_weights(data=df_train, params=data_generation_parameter)
df_test = create_data_with_ipc_weights(data=df_test, params=data_generation_parameter)

### Random Forest Modell ###
# Fit
params_rf["random_state"] = seed
clf = DecisionTreeBaggingClassifier(params_rf)
clf.fit(
    X=df_train.drop(
        ["time", "event", "weights_ipcw", "survived"], axis=1
    ).values,
    y=df_train["survived"].values,
    sample_weights=df_train["weights_ipcw"].values,
)

# Evaluation auf Testdaten
_ , pred  =clf.predict_proba(df_test.drop(
    ["weights_ipcw", "survived", "time", "event"], axis=1
).values)
rf_test_mse = ipc_weighted_mse(
    y_true=df_test["survived"].values,
    y_pred=pred,
    sample_weight=df_test["weights_ipcw"].values,
)

# Prediction für X_erwartung
_ ,rf_pred = clf.predict_proba(data_generation_parameter['X_pred_point'].values)


In [96]:
def calculate_ijk_butt_variance(
    clf: DecisionTreeBaggingClassifier, X_pred_point: pd.DataFrame, df_train: pd.DataFrame
) -> float:

    T_N_b, pred = clf.predict_proba(X_pred_point.values)
    N_bi = clf.nbi
    weights = df_train["weights_ipcw"]
    B, n = N_bi.shape

    cov_i = ((N_bi - n * weights.values.reshape(1,-1)).T @ (T_N_b - pred)) / B
    cov_i_hoch2 = cov_i**2
    array = cov_i_hoch2/weights.values.reshape(-1,1)

    biased_var_estimate = np.sum(array[~np.isnan(array) & ~np.isinf(array)], axis=0) * np.sum(weights**2)

    bias_correction = n / B * np.sum(1-weights[weights > 0]) * np.var(T_N_b, axis=0, ddof=1)* np.sum(weights**2)

    _ = np.var((N_bi - n * weights.values.reshape(1,-1)) * (T_N_b-pred) / B,axis=0, ddof=1) / weights.values.reshape(1,-1)
    bias_correction2 = np.sum(_[~np.isnan(_) & ~np.isinf(_)]) * np.sum(weights**2)  

    return biased_var_estimate , bias_correction[0], bias_correction2

calculate_ijk_butt_variance(clf, X_pred_point, df_train)

(0.006872117189061649, 0.0009693067120433108, 9.696912753319596e-08)

In [97]:
T_N_b, pred = clf.predict_proba(X_pred_point.values)
N_bi = clf.nbi
weights = df_train["weights_ipcw"]
B, n = N_bi.shape

cov_i = ((N_bi - n * weights.values.reshape(1,-1)).T @ (T_N_b - pred)) / B
cov_i_hoch2 = cov_i**2
array = cov_i_hoch2/weights.values.reshape(-1,1)

biased_var_estimate = np.sum(array[~np.isnan(array) & ~np.isinf(array)], axis=0) * np.sum(weights**2)

bias_correction = n / B * np.sum(1-weights[weights > 0]) * np.var(T_N_b, axis=0, ddof=1)* np.sum(weights**2)


cov_i__ =  (((N_bi - n * weights.values.reshape(1,-1))**2).T @ (T_N_b - pred)**2) / B

aa = ((weights * (1-weights) * n * np.var(T_N_b, axis=0, ddof=1)).values.reshape(-1,1) + cov_i__ - cov_i_hoch2 ) / weights.values.reshape(-1,1)

bias_correction2 = np.sum(weights**2) * 1/B  * np.sum(aa[~np.isnan(aa) & ~np.isinf(aa)]) 

biased_var_estimate , bias_correction[0], bias_correction2



(0.006872117189061649, 0.0009693067120433108, 0.0019389010182477378)

array([[ 0.],
       [ 0.],
       [nan],
       [ 0.],
       [nan],
       [nan],
       [ 0.]])

In [69]:
weights

0    0.285714
1    0.285714
2    0.000000
3    0.285714
4    0.000000
5    0.000000
6    0.142857
Name: weights_ipcw, dtype: float64

In [58]:
aa / bb

array([[nan],
       [ 1.],
       [ 1.],
       [ 1.],
       [ 1.],
       [ 1.],
       [ 1.]])